In [ ]:
import numpy as np  
import pandas as pd  
from glob import glob 
import json
import ast 
 
### MMStar questions 
human_mc=pd.read_csv('/home/work/yuna/HPA/evaluation/scored/humans/human_mc_per_question.csv') 
qids = human_mc.pid.unique() 
human_mc['pid'] = human_mc['pid'].astype('Int64')
print(len(qids))

247


In [68]:
import seaborn as sns 
from utils.score import get_summary  

sns.set_theme(
    style="whitegrid",
    context="paper",
    font="serif"
)
model_results = {} 
model_mc = get_summary('mmstar')
model_mc['pid'] = model_mc['pid'].astype('Int64') 
model_mc = model_mc.drop_duplicates(subset=['model', 'pid', 'condition']) 
model_mc=model_mc[model_mc['condition'] == 'inst blind']
print(model_mc.condition.unique())
mm = pd.merge(model_mc, human_mc, on =['pid', 'l2_category', 'category', 'answer'], how='left', suffixes=('_model', '_human')) 
mm = mm[(mm['pid'].isin(qids))]
mm = mm[['question_model', 'answer', 'category', 'l2_category', 'output_model', 'extracted_choice', 'correct', 
       'subject_agreements', 'agreement', 'model', 'answers', 'extracted_choices',
       'pid', 'mean_accuracy', 'std_accuracy',  'accuracies',  'confidences', 'mean_confidence', 'std_confidence', 
       'question_human',  'human_qid', 'original_question', 'meta_info_model', ]]

89
['inst blind']


In [96]:
human_answers = {}
for i, row in human_mc.iterrows(): 
    human_answers[row['pid']] = ast.literal_eval(row['extracted_choices'])[:20]
human_choices = pd.DataFrame.from_dict(human_answers, orient='index')
human_choices.columns = [data['participant_id'] for data in ast.literal_eval(mm.iloc[0]['subject_agreements'])]
human_choices.index.name = "pid"
human_choices.reset_index(inplace=True)
human_choices.head()

,pid,02f952bb_20251210_230336,3a36e078_20251205_225238,5d843d39_20251127_141921,6b3a1a8d_20251125_135446,9c5d09f3_20251209_121033,13f54aa2_20251204_125847,14d2420f_20251208_150302,60de6bd6_20251210_170035,73df43f3_20251204_210431,...,0756a860_20251124_133827,825ff3e9_20251210_192335,3238ebaf_20251126_140604,29127478_20251202_100107,a5909497_20251211_232822,b9b2cb98_20251127_193952,d1548553_20251125_202324,dd0f9986_20251128_222537,f616d5c7_20251203_212223,2e184452_20251205_141511_cleaned
0,618,B,B,B,B,A,C,B,A,B,...,B,A,A,A,B,A,A,A,A,B
1,189,A,A,C,A,B,D,B,A,A,...,B,B,C,B,A,C,A,C,B,D
2,614,D,D,B,B,B,D,D,B,B,...,D,B,B,A,B,D,D,A,B,B
3,812,B,A,A,A,C,C,A,B,D,...,A,B,C,C,D,A,B,D,B,C
4,597,D,C,D,C,D,B,C,D,D,...,D,A,D,C,C,C,C,A,A,D


In [109]:
human_choices.isna().sum()

pid                                 0
02f952bb_20251210_230336            0
3a36e078_20251205_225238            0
5d843d39_20251127_141921            0
6b3a1a8d_20251125_135446            0
9c5d09f3_20251209_121033            0
13f54aa2_20251204_125847            0
14d2420f_20251208_150302            0
60de6bd6_20251210_170035            0
73df43f3_20251204_210431            0
218b2db5_20251126_084136            0
0756a860_20251124_133827            0
825ff3e9_20251210_192335            0
3238ebaf_20251126_140604            0
29127478_20251202_100107            0
a5909497_20251211_232822            0
b9b2cb98_20251127_193952            0
d1548553_20251125_202324            0
dd0f9986_20251128_222537            0
f616d5c7_20251203_212223            0
2e184452_20251205_141511_cleaned    2
dtype: int64

In [98]:
pts = [data['participant_id'] for data in ast.literal_eval(mm.iloc[0]['subject_agreements'])]

In [108]:
from sklearn.metrics import cohen_kappa_score

subj_agreement = { }
for pt in pts :
    subj_agreement[pt] = []
    target = human_choices[pt].values  
    for jpt in pts :
        if jpt != pt : 
            try:
                kappa = cohen_kappa_score(target, human_choices[jpt].values)
                subj_agreement[pt].append(kappa)
            except Exception as e: 
                # print(jpt) # , target, human_choices[jpt].values)
                pass 
                
    print(len(subj_agreement[pt]))

18
18
18
18
18
18
18
18
18
18
18
18
18
18
18
18
18
18
18
0


In [97]:
for model in model_mc.model.unique(): 
    print(model)
    df = model_mc[model_mc['model'] == model]
    df = pd.merge(df, human_choices, on='pid', how='inner')
    print(len(df))

InternVL3_5-1B
247
InternVL3_5-2B
247
InternVL3_5-4B
247
InternVL3_5-8B
247
Qwen3-VL-2B-Instruct
247
Qwen3-VL-4B-Instruct
247
Qwen3-VL-8B-Instruct
247
llava-v1.6-mistral-7b-hf
247
InternVL3_5-8B_A1_vqa_gt
247
InternVL3_5-8B_SFT_mmstar_15_blind_inst
247
InternVL3_5-8B_A2_vqa_10_blind_inst
247
InternVL3_5-8B_A3_vqa_15_blind_inst
247
InternVL3_5-8B_A4_mmstar_15_blind_inst
247
InternVL3_5-8B_SFT_vqa_15_blind_inst
247
InternVL3_5-8B_SFT_vqa_gt
247
Qwen3-VL-4B-Instruct_A1_vqa_gt
247
Qwen3-VL-4B-Instruct_A2_vqa_10_blind_inst
247
Qwen3-VL-4B-Instruct_A3_vqa_15_blind_inst
247
Qwen3-VL-4B-Instruct_A4_mmstar_15_blind_inst
247
Qwen3-VL-4B-Instruct_SFT_vqa_15_blind_inst
247
Qwen3-VL-4B-Instruct_SFT_vqa_gt
247
Qwen3-VL-8B-Instruct_A1_vqa_gt
247
Qwen3-VL-8B-Instruct_A2_vqa_10_blind_inst
247
Qwen3-VL-8B-Instruct_A3_vqa_15_blind_inst
247
Qwen3-VL-8B-Instruct_A4_mmstar_15_blind_inst
247
Qwen3-VL-8B-Instruct_SFT_mmstar_15_blind_inst
247
Qwen3-VL-8B-Instruct_SFT_vqa_15_blind_inst
247
llava-v1.6-mistral-7b

In [7]:
pt = model_mc.pivot_table( 
    index=['model', 'pid', 'category', 'l2_category'],  
    columns=['condition'],  
    values=['correct'],
    aggfunc=['mean'] # , 'count' 
)
pt['MG'] = pt[('mean', 'correct', '')] - pt[('mean', 'correct', 'inst blind')] 
pt['inst_acc'] = pt[('mean', 'correct', 'inst blind')] - pt[('mean', 'correct', 'blind')]
pt.to_csv(f'./tables/mmstar_model_MG_by_qid.csv') 
pt.columns = ['_'.join([str(i) for i in col if str(i) != '']).strip('_') for col in pt.columns.values]
pt = pt.reset_index()  
 

In [66]:
pt = pt.pivot_table( 
    index=['model'], 
    # columns=['category', 'l2_category'],  # 'condition',  
    values=['mean_correct', 'mean_correct_inst blind', 'inst_acc', 'MG'],
    aggfunc=['mean'] # , 'count' 
)  

pt.rename(columns={
    "mean_correct_inst blind": "Blind_inst",
    "inst_acc": "Delta_inst",
    "mean_correct_blind": "Blind",
    "mean_correct": "GT" 
}, inplace=True) 
pt.dropna(axis=0).to_latex(f'./tables/A2_mmstar_mg_inst.tex', float_format="%.1f")   

mmstar_human_comparison = pd.concat([model_mc[model_mc['pid'].isin(qids)], human_mc]) 
pt = mmstar_human_comparison
pt = pt.round(3)
pt.to_csv(f'./tables/mmstar_human_comparison.csv')
with pd.option_context('display.float_format', '{:0.3f}'.format):
    display(pt)   

mean_correct = mmstar_human_comparison.groupby(['model', 'condition'])['correct'].count()
mean_correct # .to_latex(f'./tables/mmstar_human_comparison_pretrained.tex', float_format="%.1f")   

KeyError: 'mean_correct'

In [ ]:
mmstar_results = analyze_mmstar_by_categories(human_mc, model_mc_agg)
print(mmstar_results['correlation_by_l2_category'])
corr_df = correlation_by_metadata(
    model_data, vqa_metadata, 'answer_type',
    x_col='human_accuracy', y_col='model_accuracy'
)
pt = pt.pivot_table( 
    index=['model'],  
    columns=['category', 'l2_category'],  
    values=['MG'],
    aggfunc=['mean'] # , 'count' 
)
pt.dropna(subset=['mean_correct', 'MG', 'inst_acc'], axis=0).dropna(axis=1) 
pt.columns = ['_'.join([str(i) for i in col if str(i) != '']).strip('_') for col in pt.columns.values] 
pt.round(3).to_csv(f'./tables/mmstar_model_MG_by_category.csv') 

In [99]:
results = pd.merge(df.groupby(["category",	"l2_category"]).agg({'correct': 'mean', 'question':'nunique'}).reset_index(),
        blind.groupby(["category",	"l2_category"])['correct'].mean().reset_index(), 
        on=["category",	"l2_category"], suffixes=('_human', '_model')
        )
        
results['diff'] = np.abs(results['correct_model'] - results['correct_human'])
results.sort_values(by=['diff']) 

,category,l2_category,correct_human,question,correct_model,diff
8,instance reasoning,single-instance reasoning,0.343750,4,0.340909,0.002841
13,math,numeric commonsense and calculation,0.250000,1,0.242857,0.007143
10,logical reasoning,common reasoning,0.240385,13,0.232877,0.007508
3,fine-grained perception,localization,0.125000,1,0.083333,0.041667
2,coarse perception,image style & quality,0.250000,2,0.291667,0.041667
0,coarse perception,image emotion,0.375000,4,0.289474,0.085526
15,science & technology,biology & chemistry & physics,0.187500,2,0.100000,0.087500
5,fine-grained perception,recognition,0.321429,7,0.208333,0.113095
11,logical reasoning,diagram reasoning,0.166667,6,0.280488,0.113821
12,math,geometry,0.270000,25,0.390000,0.120000


In [84]:
blind = mmstar[mmstar['blind'] == True]
blind = blind.rename(columns={"index": 'qid'})
blind['qid'] = blind['qid'].astype(int)
blind = blind[blind['qid'].isin(df.qid.unique())]
len(blind.qid.unique())

# Model resutls 

In [ ]:
results_by_categories = pd.merge(
    mmstar[mmstar['blind'] == False].groupby(['category', 'l2_category']).correct.mean().reset_index().sort_values(by=['correct'], ascending=False), 
    mmstar[mmstar['blind'] == True].groupby(['category', 'l2_category']).correct.mean().reset_index().sort_values(by=['correct'], ascending=False), 
    on=['category', 'l2_category'], 
    how='inner', 
    suffixes=('', '_blind') 
)

results_by_categories['diff'] = results_by_categories['correct'] - results_by_categories['correct_blind'] 
results_by_categories.sort_values(by=['diff'], ascending=False)

,category,l2_category,correct,correct_blind,diff
0,coarse perception,image style & quality,0.610357,0.301907,0.308450
3,instance reasoning,single-instance reasoning,0.560606,0.259871,0.300735
2,coarse perception,image emotion,0.562105,0.308901,0.253205
7,fine-grained perception,localization,0.438333,0.189583,0.248750
1,instance reasoning,cross-instance relation reasoning,0.563364,0.318182,0.245182
6,coarse perception,image scene and topic,0.443238,0.220917,0.222321
5,logical reasoning,diagram reasoning,0.453247,0.235450,0.217797
4,instance reasoning,cross-instance attribute reasoning,0.462279,0.269663,0.192616
8,logical reasoning,common reasoning,0.433522,0.250476,0.183046
10,math,statistical reasoning,0.399502,0.234884,0.164618


In [19]:
qs = mmstar[mmstar['blind'] == True].groupby(['category', 'l2_category', 'question', 'answer']).correct.mean().reset_index().sort_values(by=['correct'], ascending=False)
qs.index.name = 'qid'
qs.to_csv('./csv/mmstar_questions_blind_desceanding.csv')
qs = qs.reset_index() 

In [15]:
mmstar.groupby(['category', 'l2_category']).question.nunique()

category                 l2_category                            
coarse perception        image emotion                               19
                         image scene and topic                      135
                         image style & quality                       21
fine-grained perception  localization                                39
                         object counting                             92
                         recognition                                118
instance reasoning       cross-instance attribute reasoning          75
                         cross-instance relation reasoning           48
                         single-instance reasoning                   93
logical reasoning        code & sequence reasoning                   38
                         common reasoning                            97
                         diagram reasoning                          108
math                     geometry                                   116

In [5]:
mmstar = [] 

for file in glob('../results/mmstar/*.jsonl'): 
    
    with open(file, 'r') as f:
        data = [json.loads(line) for line in f]
    df = pd.DataFrame(data)
    df['model'] = '_'.join(file.replace('_224', '').split('/')[-1].split('.jsonl')[0].split('_')[1:])
    mmstar.append(df) 

mmstar = pd.concat(mmstar, axis=0) 
mmstar['blind'] = mmstar['model'].str.contains('-blind') 
mmstar['model'] = mmstar['model'].str.replace('-blind', '') 
mmstar['correct'] = mmstar['output'].str.strip().str.lower() == mmstar['answer'].str.strip().str.lower()
results = mmstar.groupby(['model', 'category', 'blind'])['correct'].mean().reset_index().pivot(index=['model', 'blind'], columns='category', values='correct') 
print(results.dropna(axis=0).reset_index().model.unique())
results 

['InternVL2_5-1B' 'InternVL2_5-2B' 'InternVL2_5-4B' 'InternVL2_5-8B'
 'InternVL3-1B' 'InternVL3-2B' 'Qwen2.5-VL-3B-Instruct'
 'Qwen2.5-VL-7B-Instruct' 'Qwen3-VL-4B-Instruct' 'Qwen3-VL-8B-Instruct'
 'gemma-3-12b-it' 'gemma-3-4b-it']


category                             coarse perception  \
model                         blind                      
InternVL2_5-1B                False              0.538   
                              True               0.256   
InternVL2_5-2B                False              0.676   
                              True               0.448   
InternVL2_5-4B                False              0.708   
                              True               0.284   
InternVL2_5-8B                False              0.352   
                              True               0.000   
InternVL3-1B                  False              0.448   
                              True               0.448   
InternVL3-2B                  False              0.488   
                              True               0.272   
Qwen2-VL-2B-Instruct-bnb-4bit False              0.000   
                              True               0.000   
Qwen2-VL-7B-Instruct-bnb-4bit False              0.000   
                              True               0.000   
Qwen2.5-VL-3B-Instruct        False              0.456   
                              True               0.324   
Qwen2.5-VL-7B-Instruct        False              0.500   
                              True               0.260   
Qwen3-VL-4B-Instruct          False              0.712   
Qwen3-VL-8B-Instruct          False              0.752   
                              True               0.256   
gemma-3-12b-it                False              0.704   
                              True               0.340   
gemma-3-4b-it                 False              0.640   
                              True               0.304   

category                             fine-grained perception  \
model                         blind                            
InternVL2_5-1B                False                    0.326   
                              True                     0.200   
InternVL2_5-2B                False                    0.404   
                              True                     0.320   
InternVL2_5-4B                False                    0.512   
                              True                     0.180   
InternVL2_5-8B                False                    0.350   
                              True                     0.000   
InternVL3-1B                  False                    0.388   
                              True                     0.388   
InternVL3-2B                  False                    0.424   
                              True                     0.240   
Qwen2-VL-2B-Instruct-bnb-4bit False                    0.000   
                              True                     0.000   
Qwen2-VL-7B-Instruct-bnb-4bit False                      NaN   
                              True                       NaN   
Qwen2.5-VL-3B-Instruct        False                    0.364   
                              True                     0.196   
Qwen2.5-VL-7B-Instruct        False                    0.420   
                              True                     0.240   
Qwen3-VL-4B-Instruct          False                    0.480   
Qwen3-VL-8B-Instruct          False                    0.560   
                              True                     0.192   
gemma-3-12b-it                False                    0.416   
                              True                     0.232   
gemma-3-4b-it                 False                    0.340   
                              True                     0.244   

category                             instance reasoning  logical reasoning  \
model                         blind                                          
InternVL2_5-1B                False               0.458           0.268000   
                              True                0.288           0.224000   
InternVL2_5-2B                False               0.608           0.424000   
                              True                0.420           0.28